In [1]:
import os
import gc
import time
import warnings

import torch
import pandas as pd

from tqdm.auto import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq,
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)

warnings.filterwarnings("ignore")


# ============================================================
# CONFIGURATION
# ============================================================

TRAIN_PATH = r"fewshot_examples_17_set3.csv"
TEST_PATH  = r"P_CULTA_V2.csv"

MODEL_ID = "meta-llama/Meta-Llama-3.1-8B-Instruct"

NUM_EPOCHS = 10

# Same general token setup as your previous generation code
MAX_LENGTH = 2048
MAX_NEW_TOKENS = 40


# ============================================================
# QLoRA CONFIGURATION
# ============================================================

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05


# ============================================================
# TRAINING CONFIGURATION
# ============================================================

BATCH_SIZE = 1
GRADIENT_ACCUMULATION = 8

LEARNING_RATE = 2e-4


# ============================================================
# LOAD DATA
# ============================================================

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print("==============================================")
print("DATASET")
print("==============================================")

print(f"Train shape : {train_df.shape}")
print(f"Test shape  : {test_df.shape}")

print("\nColumns:")
print(train_df.columns.tolist())

print("\n==============================================\n")


# ============================================================
# CHECK REQUIRED COLUMNS
# ============================================================

required_columns = [
    "User Utterance",
    "Context",
    "User Role",
    "Model Role",
    "Power Distance",
    "Gold Response",
]

for col in required_columns:

    if col not in train_df.columns:

        raise ValueError(
            f"Missing column in training file: {col}"
        )

    if col != "Gold Response" and col not in test_df.columns:

        raise ValueError(
            f"Missing column in test file: {col}"
        )


# ============================================================
# GPU CHECK
# ============================================================

if not torch.cuda.is_available():

    raise RuntimeError(
        "CUDA GPU not available."
    )


print("\n================ GPU INFO ================")

print(
    f"GPU : {torch.cuda.get_device_name(0)}"
)

props = torch.cuda.get_device_properties(0)

print(
    f"Total VRAM : "
    f"{props.total_memory / 1024**3:.2f} GB"
)

print(
    f"Allocated : "
    f"{torch.cuda.memory_allocated() / 1024**3:.2f} GB"
)

print(
    f"Reserved  : "
    f"{torch.cuda.memory_reserved() / 1024**3:.2f} GB"
)

print("==========================================\n")


# ============================================================
# SYSTEM INSTRUCTION
# ============================================================
#
# This is intentionally the SAME instruction used
# during your prompting experiments.
#
# There are NO demonstrations here.
#
# ============================================================

SYSTEM_INSTRUCTION = (
    "Generate a natural Urdu response. "
    "Output only the response utterance. "
    "Do not explain. "
    "Do not narrate. "
    "Do not add extra context. "
    "Do not ask unnecessary follow-up questions."
)


# ============================================================
# 4-BIT QUANTIZATION
# ============================================================

bnb_config = BitsAndBytesConfig(

    load_in_4bit=True,

    bnb_4bit_use_double_quant=True,

    bnb_4bit_quant_type="nf4",

    bnb_4bit_compute_dtype=torch.float16,
)


# ============================================================
# MEMORY PRINT FUNCTION
# ============================================================

def print_memory(title):

    print(
        f"\n================ {title} ================"
    )

    print(
        f"Allocated : "
        f"{torch.cuda.memory_allocated() / 1024**3:.2f} GB"
    )

    print(
        f"Reserved  : "
        f"{torch.cuda.memory_reserved() / 1024**3:.2f} GB"
    )

    print(
        f"Max Allocated : "
        f"{torch.cuda.max_memory_allocated() / 1024**3:.2f} GB"
    )

    print(
        f"Max Reserved  : "
        f"{torch.cuda.max_memory_reserved() / 1024**3:.2f} GB"
    )

    print("==========================================\n")


# ============================================================
# GPU CLEANUP
# ============================================================

def cleanup_gpu():

    gc.collect()

    if torch.cuda.is_available():

        torch.cuda.empty_cache()

        try:

            torch.cuda.ipc_collect()

        except Exception:

            pass


# ============================================================
# LOAD FRESH QWEN MODEL
# ============================================================
#
# IMPORTANT:
# A completely fresh Qwen model is loaded for every
# experiment.
#
# trust_remote_code=False prevents Transformers from
# trying to download custom_generate/generate.py.
#
# ============================================================

def load_fresh_model():

    print("\nLoading FRESH Qwen model...")

    # --------------------------------------------------------
    # MODEL
    # --------------------------------------------------------

    model = AutoModelForCausalLM.from_pretrained(

        MODEL_ID,

        quantization_config=bnb_config,

        device_map="auto",

        # IMPORTANT FIX
        trust_remote_code=False,
    )

    # --------------------------------------------------------
    # TOKENIZER
    # --------------------------------------------------------

    tokenizer = AutoTokenizer.from_pretrained(

        MODEL_ID,

        # IMPORTANT FIX
        trust_remote_code=False,
    )

    # --------------------------------------------------------
    # PAD TOKEN
    # --------------------------------------------------------

    if tokenizer.pad_token is None:

        tokenizer.pad_token = tokenizer.eos_token

    model.config.pad_token_id = tokenizer.pad_token_id

    # --------------------------------------------------------
    # PREPARE 4-BIT MODEL FOR TRAINING
    # --------------------------------------------------------

    model = prepare_model_for_kbit_training(
        model
    )

    # --------------------------------------------------------
    # LoRA
    # --------------------------------------------------------

    lora_config = LoraConfig(

        r=LORA_R,

        lora_alpha=LORA_ALPHA,

        lora_dropout=LORA_DROPOUT,

        target_modules=[
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
            "gate_proj",
            "up_proj",
            "down_proj",
        ],

        bias="none",

        task_type="CAUSAL_LM",
    )

    model = get_peft_model(

        model,

        lora_config,
    )

    # --------------------------------------------------------
    # TRAINABLE PARAMETERS
    # --------------------------------------------------------

    model.print_trainable_parameters()

    print_memory(
        "MEMORY AFTER MODEL LOAD"
    )

    return model, tokenizer


# ============================================================
# BUILD USER CONTENT
# ============================================================

def build_user_content(
    row,
    input_columns,
):

    parts = []

    for col in input_columns:

        value = row[col]

        if pd.isna(value):

            value = ""

        value = str(value).strip()

        parts.append(
            f'{col}: "{value}"'
        )

    return "\n\n".join(parts)


# ============================================================
# PREPARE SFT DATA
# ============================================================
#
# TRAINING FORMAT:
#
# SYSTEM
# USER
# ASSISTANT = GOLD RESPONSE
#
# Loss is calculated ONLY on the response.
#
# ============================================================

def prepare_training_dataset(
    df,
    input_columns,
    tokenizer,
):

    dataset = []

    max_total_tokens = 0

    max_response_tokens = 0

    print(
        "\nBuilding training examples..."
    )

    for _, row in tqdm(

        df.iterrows(),

        total=len(df),

        desc="Preparing SFT data",

    ):

        # ----------------------------------------------------
        # USER INPUT
        # ----------------------------------------------------

        user_content = build_user_content(

            row,

            input_columns,
        )

        # ----------------------------------------------------
        # GOLD RESPONSE
        # ----------------------------------------------------

        gold_response = row[
            "Gold Response"
        ]

        if pd.isna(gold_response):

            gold_response = ""

        gold_response = str(
            gold_response
        ).strip()

        # ----------------------------------------------------
        # PROMPT ONLY
        # ----------------------------------------------------

        prompt_messages = [

            {
                "role": "system",
                "content": SYSTEM_INSTRUCTION,
            },

            {
                "role": "user",
                "content": user_content,
            },
        ]

        prompt_text = tokenizer.apply_chat_template(

            prompt_messages,

            tokenize=False,

            add_generation_prompt=True,
        )

        # ----------------------------------------------------
        # FULL TRAINING EXAMPLE
        # ----------------------------------------------------

        full_messages = [

            {
                "role": "system",
                "content": SYSTEM_INSTRUCTION,
            },

            {
                "role": "user",
                "content": user_content,
            },

            {
                "role": "assistant",
                "content": gold_response,
            },
        ]

        full_text = tokenizer.apply_chat_template(

            full_messages,

            tokenize=False,

            add_generation_prompt=False,
        )

        # ----------------------------------------------------
        # TOKENIZE PROMPT
        # ----------------------------------------------------

        prompt_tokens = tokenizer(

            prompt_text,

            add_special_tokens=False,

        )["input_ids"]

        prompt_length = len(
            prompt_tokens
        )

        # ----------------------------------------------------
        # TOKENIZE FULL SEQUENCE
        # ----------------------------------------------------

        full_tokens = tokenizer(

            full_text,

            add_special_tokens=False,

            truncation=True,

            max_length=MAX_LENGTH,
        )

        input_ids = full_tokens[
            "input_ids"
        ]

        attention_mask = full_tokens[
            "attention_mask"
        ]

        # ----------------------------------------------------
        # LABELS
        #
        # Prompt tokens = -100
        #
        # Gold response tokens = actual token IDs
        #
        # Therefore loss is only calculated on response.
        # ----------------------------------------------------

        labels = []

        for token_index in range(
            len(input_ids)
        ):

            if token_index < prompt_length:

                labels.append(-100)

            else:

                labels.append(
                    input_ids[token_index]
                )

        # ----------------------------------------------------
        # STATISTICS
        # ----------------------------------------------------

        response_length = max(

            0,

            len(input_ids) - prompt_length
        )

        max_total_tokens = max(

            max_total_tokens,

            len(input_ids)
        )

        max_response_tokens = max(

            max_response_tokens,

            response_length
        )

        # ----------------------------------------------------
        # ADD EXAMPLE
        # ----------------------------------------------------

        dataset.append({

            "input_ids": input_ids,

            "attention_mask": attention_mask,

            "labels": labels,

        })

    # --------------------------------------------------------
    # PRINT STATISTICS
    # --------------------------------------------------------

    print(
        f"\nTraining examples : "
        f"{len(dataset)}"
    )

    print(
        f"Maximum total tokens : "
        f"{max_total_tokens}"
    )

    print(
        f"Maximum response tokens : "
        f"{max_response_tokens}"
    )

    print(
        f"MAX_LENGTH : "
        f"{MAX_LENGTH}"
    )

    return dataset


# ============================================================
# PYTORCH DATASET
# ============================================================

class SFTDataset(
    torch.utils.data.Dataset
):

    def __init__(
        self,
        data,
    ):

        self.data = data

    def __len__(self):

        return len(self.data)

    def __getitem__(
        self,
        idx,
    ):

        return self.data[idx]


# ============================================================
# GENERATE TEST RESPONSES
# ============================================================

def generate_test_responses(

    model,

    tokenizer,

    test_df,

    input_columns,

    output_path,

):

    model.eval()

    responses = []

    max_tokens_seen = 0

    print(
        "\n================================================"
    )

    print(
        "GENERATING TEST RESPONSES"
    )

    print(
        f"Input columns: {input_columns}"
    )

    print(
        f"Test samples: {len(test_df)}"
    )

    print(
        "================================================\n"
    )

    for i, row in tqdm(

        test_df.iterrows(),

        total=len(test_df),

        desc="Generation",

    ):

        # ----------------------------------------------------
        # BUILD INPUT
        # ----------------------------------------------------

        user_content = build_user_content(

            row,

            input_columns,
        )

        # ----------------------------------------------------
        # TEST PROMPT
        # ----------------------------------------------------

        messages = [

            {
                "role": "system",

                "content":
                    SYSTEM_INSTRUCTION,
            },

            {
                "role": "user",

                "content":
                    user_content,
            },
        ]

        # ----------------------------------------------------
        # CHAT TEMPLATE
        # ----------------------------------------------------

        text_in = tokenizer.apply_chat_template(

            messages,

            tokenize=False,

            add_generation_prompt=True,
        )

        # ----------------------------------------------------
        # TOKEN COUNT
        # ----------------------------------------------------

        num_tokens = len(

            tokenizer(
                text_in
            )["input_ids"]
        )

        max_tokens_seen = max(

            max_tokens_seen,

            num_tokens,
        )

        # ----------------------------------------------------
        # TOKENIZE
        # ----------------------------------------------------

        inputs = tokenizer(

            text_in,

            return_tensors="pt",

            truncation=True,

            max_length=MAX_LENGTH,
        )

        # Move inputs to model's device
        inputs = {
            key: value.to(model.device)
            for key, value in inputs.items()
        }

        # ----------------------------------------------------
        # GENERATION
        # ----------------------------------------------------

        with torch.no_grad():

            if i % 10 == 0:

                print(

                    f"\nBefore generate : "

                    f"{torch.cuda.memory_allocated()/1024**3:.2f} GB allocated | "

                    f"{torch.cuda.memory_reserved()/1024**3:.2f} GB reserved"
                )

            outputs = model.generate(

                **inputs,

                max_new_tokens=MAX_NEW_TOKENS,

                temperature=0.3,

                do_sample=True,

                repetition_penalty=1.1,

                pad_token_id=
                    tokenizer.eos_token_id,

                use_cache=True,
            )

            if i % 10 == 0:

                print(

                    f"After generate  : "

                    f"{torch.cuda.memory_allocated()/1024**3:.2f} GB allocated | "

                    f"{torch.cuda.memory_reserved()/1024**3:.2f} GB reserved"
                )

        # ----------------------------------------------------
        # REMOVE INPUT TOKENS
        # ----------------------------------------------------

        new_tokens = outputs[

            0

        ][

            inputs["input_ids"].shape[1]:
        ]

        # ----------------------------------------------------
        # DECODE RESPONSE
        # ----------------------------------------------------

        response = tokenizer.decode(

            new_tokens,

            skip_special_tokens=True,
        ).strip()

        responses.append(
            response
        )

        # ----------------------------------------------------
        # FREE MEMORY
        # ----------------------------------------------------

        del outputs

        del new_tokens

        del inputs

        gc.collect()

        torch.cuda.empty_cache()

        # ----------------------------------------------------
        # DIAGNOSTICS
        # ----------------------------------------------------

        if i % 10 == 0:

            print(
                "\n----------------------------------------"
            )

            print(
                f"Sample         : {i}"
            )

            print(
                f"Prompt Tokens  : {num_tokens}"
            )

            print(
                f"Maximum So Far : {max_tokens_seen}"
            )

            print(
                f"Allocated VRAM : "
                f"{torch.cuda.memory_allocated()/1024**3:.2f} GB"
            )

            print(
                f"Reserved VRAM  : "
                f"{torch.cuda.memory_reserved()/1024**3:.2f} GB"
            )

            print(
                "----------------------------------------"
            )

        # ----------------------------------------------------
        # BACKUP EVERY 25 SAMPLES
        # ----------------------------------------------------

        if i % 25 == 0 and i > 0:

            backup = test_df.copy()

            backup[
                "LLaMA_Response"
            ] = (

                responses
                + [""] * (

                    len(test_df)
                    - len(responses)
                )
            )

            backup.to_csv(

                output_path.replace(

                    ".csv",

                    "_backup.csv",
                ),

                index=False,

                encoding="utf-8-sig",
            )

    # ========================================================
    # FINAL SAVE
    # ========================================================

    result = test_df.copy()

    result[
        "LLaMA_Response"
    ] = responses

    result.to_csv(

        output_path,

        index=False,

        encoding="utf-8-sig",
    )

    print(
        f"\nSaved -> {output_path}"
    )

    return result


# ============================================================
# RUN ONE COMPLETE SFT EXPERIMENT
# ============================================================

def run_sft_experiment(

    experiment_name,

    input_columns,

    output_path,
):

    print("\n\n")

    print("=" * 75)

    print(
        f"STARTING SFT EXPERIMENT: "
        f"{experiment_name}"
    )

    print(
        f"INPUT COLUMNS: "
        f"{input_columns}"
    )

    print(
        f"EPOCHS: "
        f"{NUM_EPOCHS}"
    )

    print("=" * 75)

    # --------------------------------------------------------
    # CLEAN GPU
    # --------------------------------------------------------

    cleanup_gpu()

    torch.cuda.reset_peak_memory_stats()

    print_memory(
        "MEMORY BEFORE MODEL LOAD"
    )

    # --------------------------------------------------------
    # FRESH MODEL
    # --------------------------------------------------------

    model, tokenizer = (
        load_fresh_model()
    )

    # --------------------------------------------------------
    # PREPARE TRAIN DATA
    # --------------------------------------------------------

    train_data = (
        prepare_training_dataset(

            train_df,

            input_columns,

            tokenizer,
        )
    )

    train_dataset = SFTDataset(
        train_data
    )

    # --------------------------------------------------------
    # DATA COLLATOR
    # --------------------------------------------------------

    data_collator = DataCollatorForSeq2Seq(

        tokenizer=tokenizer,

        padding=True,

        return_tensors="pt",
    )

    print_memory(
        "MEMORY BEFORE TRAINING"
    )

    # --------------------------------------------------------
    # TRAINING ARGUMENTS
    # --------------------------------------------------------

    training_args = TrainingArguments(

        output_dir=(
            f"./sft_{experiment_name}"
        ),

        num_train_epochs=NUM_EPOCHS,

        per_device_train_batch_size=
            BATCH_SIZE,

        gradient_accumulation_steps=
            GRADIENT_ACCUMULATION,

        learning_rate=
            LEARNING_RATE,

        fp16=True,

        optim="paged_adamw_8bit",

        logging_steps=1,

        save_strategy="no",

        report_to="none",

        remove_unused_columns=False,

        gradient_checkpointing=True,

        max_grad_norm=0.3,

        warmup_ratio=0.03,

        lr_scheduler_type="cosine",
    )

    # --------------------------------------------------------
    # TRAINER
    # --------------------------------------------------------

    trainer = Trainer(

        model=model,

        args=training_args,

        train_dataset=train_dataset,

        data_collator=data_collator,
    )

    # --------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------

    print("\n")

    print(
        "================================================"
    )

    print(
        f"TRAINING {experiment_name}"
    )

    print(
        "================================================"
    )

    start_time = time.time()

    trainer.train()

    training_time = (
        time.time()
        - start_time
    )

    print(
        "\n================================================"
    )

    print(
        "TRAINING COMPLETE"
    )

    print(
        f"Training time: "
        f"{training_time / 60:.2f} minutes"
    )

    print(
        "================================================"
    )

    print_memory(
        "MEMORY AFTER TRAINING"
    )

    # --------------------------------------------------------
    # GENERATE TEST
    # --------------------------------------------------------

    result = generate_test_responses(

        model=model,

        tokenizer=tokenizer,

        test_df=test_df,

        input_columns=input_columns,

        output_path=output_path,
    )

    # --------------------------------------------------------
    # CLEANUP
    # --------------------------------------------------------

    print(
        "\nCleaning up model..."
    )

    del trainer

    del model

    del tokenizer

    del train_dataset

    del train_data

    cleanup_gpu()

    print_memory(
        "FINAL MEMORY AFTER CLEANUP"
    )

    return result


# ============================================================
# EXPERIMENT 1
# U
# ============================================================

result_U = run_sft_experiment(

    experiment_name="U",

    input_columns=[
        "User Utterance"
    ],

    output_path=(
        r"Set3_SFT_U_llama_test.csv"
    ),
)


# ============================================================
# EXPERIMENT 2
# U + CONTEXT
# ============================================================

result_UC = run_sft_experiment(

    experiment_name="U_C",

    input_columns=[
        "User Utterance",
        "Context",
    ],

    output_path=(
        r"Set3_SFT_U_C_llama_test.csv"
    ),
)


# ============================================================
# EXPERIMENT 3
# U + CONTEXT + ROLES
# ============================================================

result_UCR = run_sft_experiment(

    experiment_name="U_C_R",

    input_columns=[
        "User Utterance",
        "Context",
        "User Role",
        "Model Role",
    ],

    output_path=(
        r"Set3_SFT_U_C_R_llama_test.csv"
    ),
)


# ============================================================
# EXPERIMENT 4
# U + CONTEXT + ROLES + POWER DISTANCE
# ============================================================

result_UCRPD = run_sft_experiment(

    experiment_name="U_C_R_PD",

    input_columns=[
        "User Utterance",
        "Context",
        "User Role",
        "Model Role",
        "Power Distance",
    ],

    output_path=(
        r"Set3_SFT_U_C_R_PD_llama_test.csv"
    ),
)


# ============================================================
# DONE
# ============================================================

print("\n\n")

print("=" * 75)

print(
    "ALL FOUR SFT EXPERIMENTS COMPLETED"
)

print("=" * 75)

print(
    "\nGenerated files:"
)

print(
    r"1. Set3_SFT_U_llama_test.csv"
)

print(
    r"2. Set3_SFT_U_C_llama_test.csv"
)

print(
    r"3. Set3_\SFT_U_C_R_llama_test.csv"
)

print(
    r"4. Set3_SFT_U_C_R_PD_llama_test.csv"
)

print("=" * 75)

D:\stdFurqan\FYP_AA\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DATASET
Train shape : (17, 11)
Test shape  : (255, 11)

Columns:
['Language', 'Topic', 'User Role', 'Model Role', 'Power Distance', 'Register', 'Pragmatic Genre', 'Sensitivity', 'User Utterance', 'Context', 'Gold Response']



================ GPU INFO ================
GPU : NVIDIA GeForce RTX 4080 SUPER
Total VRAM : 15.99 GB
Allocated : 0.00 GB
Reserved  : 0.00 GB




STARTING SFT EXPERIMENT: U
INPUT COLUMNS: ['User Utterance']
EPOCHS: 10

================ MEMORY BEFORE MODEL LOAD ================
Allocated : 0.00 GB
Reserved  : 0.00 GB
Max Allocated : 0.00 GB
Max Reserved  : 0.00 GB


Loading FRESH Qwen model...


Loading weights: 100%|██████████| 291/291 [00:04<00:00, 64.41it/s]


trainable params: 41,943,040 || all params: 8,072,204,288 || trainable%: 0.5196

================ MEMORY AFTER MODEL LOAD ================
Allocated : 7.43 GB
Reserved  : 9.51 GB
Max Allocated : 8.25 GB
Max Reserved  : 9.51 GB


Building training examples...


Preparing SFT data: 100%|██████████| 17/17 [00:00<00:00, 617.97it/s]
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


...


Before generate : 13.35 GB allocated | 15.58 GB reserved
After generate  : 13.35 GB allocated | 15.58 GB reserved

----------------------------------------
Sample         : 240
Prompt Tokens  : 180
Maximum So Far : 249
Allocated VRAM : 13.35 GB
Reserved VRAM  : 15.58 GB
----------------------------------------


Generation:  98%|█████████▊| 250/255 [12:33<00:15,  3.01s/it]


Before generate : 13.35 GB allocated | 15.58 GB reserved
After generate  : 13.35 GB allocated | 15.59 GB reserved

----------------------------------------
Sample         : 250
Prompt Tokens  : 200
Maximum So Far : 249
Allocated VRAM : 13.35 GB
Reserved VRAM  : 15.58 GB
----------------------------------------


Generation: 100%|██████████| 255/255 [12:51<00:00,  3.03s/it]



Saved -> Set3_SFT_U_C_R_PD_llama_test.csv

Cleaning up model...

================ FINAL MEMORY AFTER CLEANUP ================
Allocated : 7.84 GB
Reserved  : 13.04 GB
Max Allocated : 15.16 GB
Max Reserved  : 15.67 GB




ALL FOUR SFT EXPERIMENTS COMPLETED

Generated files:
1. Set3_SFT_U_llama_test.csv
2. Set3_SFT_U_C_llama_test.csv
3. Set3_\SFT_U_C_R_llama_test.csv
4. Set3_SFT_U_C_R_PD_llama_test.csv
